# Curso 1: SQL Server – Nivel Básico
## Tema 1: Arquitectura y Ediciones de SQL Server

### **Objetivo del Tema**
Al finalizar este tema, comprenderás la arquitectura interna del motor de base de datos de Microsoft SQL Server, conocerás los diferentes servicios del sistema, dominarás el uso básico de la herramienta de administración SQL Server Management Studio (SSMS) y sabrás diferenciar las ediciones disponibles de SQL Server junto con sus límites de hardware y capacidad de datos.

---

## 1. El Motor de Base de Datos (Database Engine)

El **Motor de Base de Datos** (Database Engine) es el servicio central de Microsoft SQL Server que se encarga del almacenamiento físico, la seguridad, el procesamiento de datos y la transacción de la información. Se divide internamente en dos componentes principales:

### **1.1. El Motor Relacional (Relational Engine / Query Processor)**
Es el cerebro lógico del servidor. Se encarga de procesar las consultas T-SQL que envían los usuarios. Sus subcomponentes son:
*   **Parser (Analizador Sintáctico):** Verifica que la *sintaxis* de la consulta SQL sea correcta. Si hay un error de escritura (como escribir `SELEC` en vez de `SELECT`), el proceso se detiene y devuelve un error de sintaxis.
*   **Algebrizer / Binder (Algebrizador / Enlazador):** Comprueba que los objetos solicitados existan (tablas, columnas, vistas) en el catálogo de la base de datos y que el usuario tenga los tipos de datos correctos para realizar comparaciones u operaciones. Genera un árbol de consulta (*Query Tree*).
*   **Optimizer (Optimizador de Consultas):** Es uno de los componentes más complejos de SQL Server. Utiliza estadísticas de distribución de datos almacenadas en las tablas para evaluar múltiples planes de acceso física y elegir el plan con el menor **costo estimado** de CPU y Lecturas/Escrituras (I/O). El resultado es un **Plan de Ejecución**.
*   **Execution Engine (Motor de Ejecución):** Ejecuta físicamente las instrucciones contenidas en el plan de ejecución interactuando directamente con el Motor de Almacenamiento.

### **1.2. El Motor de Almacenamiento (Storage Engine)**
Es el encargado de administrar los recursos físicos y los archivos de datos en el disco y la memoria RAM. Sus subcomponentes son:
*   **Access Methods (Métodos de Acceso):** Determina la estructura física más eficiente para buscar o insertar datos (por ejemplo, mediante escaneos de tablas completas o búsquedas por índices).
*   **Buffer Manager (Administrador de Búfer):** Gestiona la memoria RAM del servidor. Todo dato que se lee o modifica en SQL Server debe estar cargado en la memoria en bloques de **8 KB llamados Páginas**. El Buffer Manager se encarga de subir páginas del disco a la memoria (**Buffer Pool**) y de escribir las páginas modificadas (páginas "sucias" o *dirty pages*) de vuelta al disco.
*   **Transaction Manager (Administrador de Transacciones):** Garantiza el cumplimiento de las propiedades **ACID** (Atomicidad, Consistencia, Aislamiento y Durabilidad). Utiliza el mecanismo **WAL (Write-Ahead Logging)**, lo que significa que cualquier cambio se escribe primero en el archivo de registro de transacciones (*Transaction Log*) en disco antes de consolidarse en el archivo de datos principal.

### **Laboratorio 1.1: Consultas de Información de Versión e Instancia**
Utiliza las siguientes consultas para diagnosticar la versión exacta del motor de SQL Server al que estás conectado y verificar sus propiedades básicas.

In [1]:
-- 1. Consultar la versión completa del motor de base de datos
SELECT @@VERSION AS [SQL_Server_Version];

(1 row affected)

SQL_Server_Version                                                                                                                                                                                      
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Microsoft SQL Server 2019 (RTM-GDR) (KB5090408) - 15.0.2170.1 (X64) 
	Apr 15 2026 15:33:32 
	Copyright (C) 2019 Microsoft Corporation
	Standard Edition (64-bit) on Windows Server 2016 Standard 10.0 <X64> (Build 14393: )

(1 row)

Total execution time: 00:00:00.124

In [1]:
-- 2. Consultar propiedades específicas de la instancia actual
SELECT 
    SERVERPROPERTY('MachineName') AS [Servidor_Fisico],
    SERVERPROPERTY('InstanceName') AS [Instancia_SQL],
    SERVERPROPERTY('Edition') AS [Edicion_Instalada],
    SERVERPROPERTY('ProductLevel') AS [Nivel_De_Producto],
    SERVERPROPERTY('ProductVersion') AS [Numero_De_Version];

(1 row affected)

Servidor_Fisico | Instancia_SQL | Edicion_Instalada         | Nivel_De_Producto | Numero_De_Version
----------------+---------------+---------------------------+-------------------+------------------
4ECF09D         | NULL          | Standard Edition (64-bit) | RTM               | 15.0.2170.1      
(1 row)

Total execution time: 00:00:00.105

---
## 2. Servicios de SQL Server

SQL Server no se ejecuta como una aplicación común de escritorio, sino como un conjunto de **servicios en segundo plano** de Windows (o Daemons en Linux). Los servicios principales y sus funciones son:

| Nombre del Servicio | Nombre Interno (Default) | Descripción |
| :--- | :--- | :--- |
| **SQL Server** | `MSSQLSERVER` | El motor de base de datos relacional propiamente dicho. Si este servicio está detenido, no se pueden realizar conexiones ni consultas a las bases de datos. |
| **SQL Server Agent** | `SQLSERVERAGENT` | El programador de tareas y automatización de SQL Server. Permite configurar y ejecutar trabajos periódicos (*Jobs*), planes de mantenimiento (backups automáticos), y enviar alertas de correo. (*No disponible en edición Express*). |
| **SQL Server Browser** | `SQLBrowser` | Resuelve solicitudes de conexión entrantes dirigidas a instancias nombradas. Escucha en el puerto UDP 1434 y redirige al cliente al puerto TCP correcto asignado dinámicamente a la instancia. |
| **SQL Server Writer** | `SQLWriter` | Permite que aplicaciones externas de respaldo (backup a nivel del S.O.) realicen copias de seguridad en caliente de las bases de datos utilizando el servicio de instantáneas de volumen de Windows (VSS). |

> **Nota de Seguridad:** Se recomienda deshabilitar el servicio *SQL Server Browser* si solo tienes una instancia predeterminada en el servidor que utiliza el puerto estándar TCP 1433, reduciendo así la superficie de ataque.

### **Laboratorio 2.1: Diagnóstico de Servicios desde T-SQL**
A partir de SQL Server 2008 R2, es posible consultar el estado y tipo de arranque de los servicios del servidor directamente desde consultas T-SQL utilizando vistas administrativas.

In [4]:
-- Versión 2019
-- Consultar los servicios activos de SQL Server instalados en la máquina anfitriona
-- Nota: Para ejecutar esto con éxito requieres el permiso de servidor 'VIEW SERVER STATE'.

SELECT
    servicename AS NombreServicio,
    status_desc AS Estado,
    startup_type_desc AS TipoInicio,
    service_account AS CuentaServicio,
    last_startup_time AS UltimoArranque,
    instant_file_initialization_enabled AS IFI
FROM sys.dm_server_services;

(4 rows affected)

NombreServicio                                     | Estado  | TipoInicio | CuentaServicio             | UltimoArranque                     | IFI
---------------------------------------------------+---------+------------+----------------------------+------------------------------------+----
SQL Server (MSSQLSERVER)                           | Running | Automatic  | NT Service\MSSQLSERVER     | 2026-06-10 22:00:20.8086847 -06:00 | N  
SQL Server Agent (MSSQLSERVER)                     | Running | Automatic  | NT Service\SQLSERVERAGENT  | NULL                               | N  
SQL Full-text Filter Daemon Launcher (MSSQLSERVER) | Running | Manual     | NT Service\MSSQLFDLauncher | NULL                               | N  
SQL Server Launchpad (MSSQLSERVER)                 | Running | Automatic  | NT Service\MSSQLLaunchpad  | NULL                               | N  
(4 rows)

Total execution time: 00:00:00.137

In [3]:
-- Versión 2025
-- Consultar los servicios activos de SQL Server instalados en la máquina anfitriona
-- Nota: Para ejecutar esto con éxito requieres el permiso de servidor 'VIEW SERVER STATE'.

SELECT 
    service_type_desc AS [Tipo_Servicio],
    display_name AS [Nombre_Visualizacion],
    status_desc AS [Estado_Servicio],
    startup_type_desc AS [Tipo_Inicio],
    process_id AS [PID],
    last_startup_time AS [Ultimo_Arranque]
FROM sys.dm_server_services;

Msg 207, Level 16, State 1, Line 6
Invalid column name 'service_type_desc'.
Msg 207, Level 16, State 1, Line 7
Invalid column name 'display_name'.

Total execution time: 00:00:00.077

---
## 3. SQL Server Management Studio (SSMS)

**SQL Server Management Studio (SSMS)** es un entorno integrado para administrar cualquier infraestructura de SQL Server, desde SQL Server local hasta Azure SQL Database. 

> **¡Concepto Clave!:** SSMS **NO** es la base de datos ni el motor de base de datos. Es solo una herramienta cliente (interfaz gráfica) que se conecta al motor de SQL Server. Es común tener SSMS instalado en tu computadora personal y conectarte de forma remota a servidores SQL Server corporativos.

### **Componentes Clave de la Interfaz:**
1.  **Object Explorer (Explorador de Objetos):** Estructura jerárquica que permite ver e interactuar con todos los objetos del servidor (bases de datos, tablas, logins, seguridad, trabajos del agente, etc.).
2.  **Query Editor (Editor de Consultas):** Espacio donde escribes scripts T-SQL. Ofrece coloreado de sintaxis, autocompletado (IntelliSense) y verificación de errores antes de la ejecución.
3.  **Resultados y Mensajes:** Paneles inferiores donde se muestran los conjuntos de datos obtenidos (pestaña *Results*) o la cantidad de filas afectadas y errores de la consulta (pestaña *Messages*).
4.  **Planes de Ejecución Gráficos:** SSMS permite solicitar el plan de ejecución estimado o real de una consulta para ver cómo el optimizador resolvió la búsqueda y cuáles operadores físicos consumieron más recursos.

### **Cadena de Conexión Básica (Connection Properties):**
*   **Server Name (Nombre de Servidor):** Indica a qué servidor conectarse.
    *   `.`, `localhost` o `127.0.0.1` representan la máquina local.
    *   `NombreServidor\NombreInstancia` para instancias nombradas (ej. `MI-PC\SQLEXPRESS`).
*   **Authentication (Autenticación):**
    *   *Windows Authentication:* Utiliza las credenciales de tu sesión de Windows actual. No requiere ingresar contraseña.
    *   *SQL Server Authentication:* Requiere un usuario interno de SQL Server (como el superusuario `sa`) y su respectiva contraseña configurada en el motor.

---
## 4. Ediciones de SQL Server y sus Límites

Microsoft ofrece SQL Server en varias ediciones adaptadas a diferentes escalas de negocio, presupuestos y necesidades técnicas. A continuación se presentan las ediciones vigentes y sus características principales:

### **4.1. Ediciones Principales**

| Característica / Límite | **Enterprise** | **Standard** | **Developer** | **Express** |
| :--- | :--- | :--- | :--- | :--- |
| **Costo** | De pago (Alto, por núcleo) | De pago (Moderado, por núcleo/CAL) | **Gratuita** | **Gratuita** |
| **Propósito** | Producción a gran escala | Producción mediana / departamental | Desarrollo y pruebas | Producción pequeña y aprendizaje |
| **Capacidad de Cómputo Máxima** | Límite del S.O. | Limitado a **24 Cores** o 4 Sockets | Límite del S.O. | Limitado a **1 CPU / 4 Cores** |
| **Memoria RAM Máxima para Búfer** | Límite del S.O. | Limitado a **128 GB** | Límite del S.O. | Limitado a **1.41 GB (1410 MB)** |
| **Tamaño Máximo de Base de Datos** | Sin límites (524 PB) | Sin límites (524 PB) | Sin límites (524 PB) | Limitado a **10 GB** por base de datos |
| **SQL Server Agent** | Incluido completo | Incluido completo | Incluido completo | **No disponible** |
| **Alta Disponibilidad** | Completa (AlwaysOn AGs de múltiples nodos) | Básica (AlwaysOn de 2 nodos, sin lecturas) | Completa (Solo con fines de prueba) | No disponible |

### **Puntos Clave a Considerar:**
1.  **Edición Developer vs Enterprise:** Tecnológicamente son **idénticas**. Todo lo que puedes hacer en Enterprise lo puedes hacer en Developer de manera gratuita. Su única restricción es la **licencia legal de uso**: Developer *nunca* debe ser usada en un entorno de producción real.
2.  **Límite de SQL Express (10 GB):** El límite de 10 GB en Express aplica estrictamente al **archivo de datos principal (`.mdf`) de cada base de datos individual**. El archivo de log de transacciones (`.ldf`) no cuenta para este límite, y puedes tener tantas bases de datos independientes de 10 GB como espacio tengas en el disco duro.
3.  **Límite de RAM de SQL Express (1.41 GB):** Aunque tu servidor físico tenga 32 GB de RAM, la instancia Express solo reservará y usará un máximo de 1.41 GB de memoria RAM para almacenar páginas de datos en caché. Esto hace que las consultas que requieran leer grandes volúmenes de datos tengan que acudir repetidamente al disco físico, reduciendo el rendimiento.

### **Laboratorio 4.1: Consultando Recursos de Almacenamiento y Memoria**
A través de estas consultas SQL, puedes medir el tamaño actual de tus bases de datos y verificar cuánta memoria física está empleando el motor de base de datos.

In [ ]:
-- 1. Consultar el tamaño total en megabytes de cada base de datos alojada en la instancia
SELECT 
    d.name AS [Base_De_Datos],
    CAST(SUM(f.size * 8 / 1024/1024) AS DECIMAL(18,2)) AS [Tamaño_Total_MB],
    COUNT(f.file_id) AS [Cantidad_Archivos_Fisicos]
FROM sys.databases d
INNER JOIN sys.master_files f ON d.database_id = f.database_id
GROUP BY d.name
ORDER BY Tamaño_Total_MB DESC;

In [ ]:
-- 2. Detalle de archivos de datos (.mdf) y log (.ldf) de la base de datos actual
SELECT 
    name AS [Nombre_Logico],
    physical_name AS [Ruta_Fisica],
    type_desc AS [Tipo_Archivo],
    CAST(size * 8 / 1024.0 AS DECIMAL(18,2)) AS [Tamaño_Actual_MB],
    max_size AS [Tamaño_Maximo_Configurado]
FROM sys.database_files;

In [8]:
-- 3. Consultar la memoria física que SQL Server está consumiendo en la memoria RAM en este momento
-- Nota: Para ejecutar esta DMV requieres permisos de VIEW SERVER STATE.
SELECT 
    physical_memory_in_use_kb / 1024 AS [Memoria_Fisica_Uso_MB],
    large_page_allocations_kb / 1024 AS [Asignacion_Paginas_Grandes_MB],
    locked_page_allocations_kb / 1024 AS [Paginas_Bloqueadas_RAM_MB],
    process_physical_memory_low AS [Alerta_Memoria_Baja]
FROM sys.dm_os_process_memory;

(1 row affected)

Memoria_Fisica_Uso_MB | Asignacion_Paginas_Grandes_MB | Paginas_Bloqueadas_RAM_MB | Alerta_Memoria_Baja
----------------------+-------------------------------+---------------------------+--------------------
16381                 | 0                             | 0                         | 0                  
(1 row)

Total execution time: 00:00:00.069

---
## 5. Autoevaluación y Ejercicios Prácticos del Tema 1

Refuerza y valida tus conocimientos teóricos con los siguientes reactivos:

### **Cuestionario de Autoevaluación**

1. **¿Qué componente de la arquitectura interna de SQL Server es responsable de elegir un plan de ejecución óptimo basado en estadísticas de distribución de datos?**
   * *Respuesta:* El **Query Optimizer (Optimizador de Consultas)**.

2. **Si estás desarrollando un sistema corporativo para una empresa y planeas utilizar SQL Server Express por temas de presupuesto, ¿qué límites de tamaño de datos y procesamiento debes contemplar obligatoriamente?**
   * *Respuesta:* Límite de procesamiento de máximo **1 CPU / 4 Cores**, un búfer máximo en memoria de **1.41 GB de RAM** y un tope estricto de **10 GB** por base de datos.

3. **Estás intentando crear un Job (tarea automática) para programar respaldos diarios en una instancia de SQL Server Express. Sin embargo, no encuentras la sección de 'SQL Server Agent' en SSMS. ¿A qué se debe esto?**
   * *Respuesta:* La edición **SQL Server Express no incluye el servicio SQL Server Agent**, por lo que no es posible utilizar Jobs programados nativos. Se debe recurrir a herramientas externas como el Programador de Tareas de Windows llamando a scripts en PowerShell o `sqlcmd`.

4. **¿Cuál es la diferencia primordial entre SQL Server Management Studio (SSMS) y el Motor de Base de Datos de SQL Server?**
   * *Respuesta:* El motor de base de datos es el servicio ejecutable que procesa y resguarda la información física. SSMS es una herramienta visual de administración de tipo cliente que interactúa con el motor mediante peticiones TCP/IP, y se puede instalar de forma separada en cualquier equipo corporativo para administrar bases de datos locales o remotas.

5. **Si una consulta SQL tiene un error de ortografía en una palabra reservada (ej. `SELECTE * FROM Tabla`), ¿cuál componente del motor intercepta este error y detiene el flujo de procesamiento?**
   * *Respuesta:* El **Parser (Analizador Sintáctico)**.

---
### **Práctica de Laboratorio Sugerida**
1. **Conexión:** Abre SQL Server Management Studio (SSMS) en tu máquina o entorno virtual y realiza la conexión a tu base de datos.
2. **Inspección de Servicios:** Abre la consola de servicios de tu sistema operativo (`services.msc`) y localiza los servicios de SQL Server. Identifica si estás usando una instancia predeterminada (`MSSQLSERVER`) o una instancia nombrada.
3. **Ejecución de Código:** Abre este notebook en tu editor compatible con Jupyter y ejecuta secuencialmente las celdas de código de laboratorio para observar en tiempo real la información de tu servidor.